# Switch timing and immediate repurchase

This notebook studies the historical timing question: if Nordea Alfa had been sold on a given date and the proceeds were immediately reinvested into another fund, what terminal outcome would that have produced by the end of the sample?

The whole fondkonto is treated as Nordea Alfa, which matches the simplifying assumption in the project README.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

candidate_dirs = [Path.cwd(), Path.cwd() / 'copilot_analysis']
analysis_dir = next(path for path in candidate_dirs if (path / 'fund_analysis.py').exists())
if str(analysis_dir) not in sys.path:
    sys.path.append(str(analysis_dir))

from fund_analysis import DEFAULT_PORTFOLIO, TaxAssumptions, load_fund_nav

In [9]:
source_fund = load_fund_nav(analysis_dir.parent / 'data' / 'nordea_alfa.csv')
target_fund_name = 'swedbank_robur_tech_a'
target_fund = load_fund_nav(analysis_dir.parent / 'data' / f'{target_fund_name}.csv')
tax = TaxAssumptions(capital_gains_tax_rate=0.30, isk_tax_rate=0.01065, isk_tax_free_amount=300000.0)

current_market_value = DEFAULT_PORTFOLIO['market_value']
cost_basis = DEFAULT_PORTFOLIO['cost_basis']
current_source_nav = source_fund['nav'].iloc[-1]
shares_held = current_market_value / current_source_nav
terminal_target_nav = target_fund['nav'].iloc[-1]
terminal_source_nav = source_fund['nav'].iloc[-1]

In [10]:
aligned = source_fund[['date', 'nav']].merge(
    target_fund[['date', 'nav']],
    on='date',
    suffixes=('_source', '_target')
)
aligned['source_market_value'] = shares_held * aligned['nav_source']
aligned['tax_paid_if_sold'] = np.maximum(aligned['source_market_value'] - cost_basis, 0) * tax.capital_gains_tax_rate
aligned['capital_after_tax'] = aligned['source_market_value'] - aligned['tax_paid_if_sold']
aligned['target_units_bought'] = aligned['capital_after_tax'] / aligned['nav_target']
aligned['terminal_value_if_switched'] = aligned['target_units_bought'] * terminal_target_nav
aligned['terminal_value_if_never_switched'] = current_market_value
aligned['switch_advantage'] = aligned['terminal_value_if_switched'] - aligned['terminal_value_if_never_switched']
aligned.tail()

,date,nav_source,nav_target,source_market_value,tax_paid_if_sold,capital_after_tax,target_units_bought,terminal_value_if_switched,terminal_value_if_never_switched,switch_advantage
1686,2026-03-06,1954.870,1327.73,779408.202793,179350.032838,600058.169955,451.942918,610696.906235,773517.0,-162820.093765
1687,2026-03-09,1923.974,1346.16,767089.943352,175654.555006,591435.388347,439.349994,593680.466818,773517.0,-179836.533182
1688,2026-03-10,1985.024,1340.87,791430.626252,182956.759876,608473.866376,453.790350,613193.286015,773517.0,-160323.713985
1689,2026-03-11,1968.206,1365.47,784725.276457,180945.154937,603780.121520,442.177508,597501.200910,773517.0,-176015.799090
1690,2026-03-12,1955.982,1351.27,779851.558066,179483.039420,600368.518646,444.299451,600368.518646,773517.0,-173148.481354


In [11]:
fig = px.line(
    aligned,
    x='date',
    y='switch_advantage',
    title=f'Historical advantage of switching from Nordea Alfa to {target_fund_name}'
)
fig.add_hline(y=0, line_dash='dash')
fig.show()

In [5]:
best_dates = aligned.nlargest(10, 'switch_advantage')[['date', 'switch_advantage', 'tax_paid_if_sold']]
worst_dates = aligned.nsmallest(10, 'switch_advantage')[['date', 'switch_advantage', 'tax_paid_if_sold']]
best_dates, worst_dates

(          date  switch_advantage  tax_paid_if_sold
 117 2019-08-14     102442.920430      79242.485557
 124 2019-08-23     102214.044038      81598.687584
 12  2019-03-12     102139.043545      80778.759420
 3   2019-02-27     102133.807859      81794.728759
 9   2019-03-07     101796.844009      80048.778154
 2   2019-02-26     101490.685370      82432.370924
 8   2019-03-06     101123.910931      81313.177952
 5   2019-03-01     100575.556847      81974.742164
 65  2019-05-29      99794.796722      82744.792859
 24  2019-03-28      99433.419457      80810.097302,
            date  switch_advantage  tax_paid_if_sold
 1738 2026-03-12    -173148.481354     179483.039420
 1729 2026-02-27    -170613.176596     193512.721979
 1730 2026-03-02    -169989.035441     189222.781277
 1731 2026-03-03    -169591.402464     182350.096762
 1736 2026-03-10    -169497.434494     182956.759876
 1737 2026-03-11    -169491.363366     180945.154937
 1735 2026-03-09    -169490.168757     175654.555006
 17

## Extensions

Repeat the notebook over several target funds, or wrap the alignment logic in a helper function once the exact scenario design is stable.